In [5]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from pathlib import Path
import pandas as pd
from xbbg import blp, Backend

In [13]:
# ============================================================
# Settings
# ============================================================

TZ = "Asia/Tokyo"
INTERVAL = 1

OUTPUT_DIR = Path(r"C:\Data\ICAM")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# Bloomberg Tickers
# ============================================================

TICKERS = {

    # --------------------------------------------------------
    # US / Japan equity futures
    # --------------------------------------------------------
    "ES": "ES1 A:00_0_R COMB Index",
    "NQ": "NQ1 A:00_0_R COMB Index",
    "RTY": "RTY1 A:00_0_R COMB Index",
    "TOPIX": "TP1 A:00_0_R COMB Index",
    "JREIT": "TRE1 A:00_0_R COMB Index",

    # --------------------------------------------------------
    # European equity futures
    # --------------------------------------------------------
    "EURO": "VG1 A:00_0_R COMB Index",
    "DAX": "GX1 A:00_0_R COMB Index",

    # --------------------------------------------------------
    # Asia equity futures
    # --------------------------------------------------------
    "TAIWAN": "TWT1 A:00_0_R COMB Index",
    "KOSPI": "KM1 A:00_0_R COMB Index",
    "CHINA_A50": "XU1 A:00_0_R COMB Index",
    "HSI": "HI1 A:00_0_R COMB Index",
    "NIFTY50": "JGS1 A:00_0_R COMB Index",

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------
    "VIX": "UX1 Index",

    # --------------------------------------------------------
    # US Treasury futures
    # --------------------------------------------------------
    "US2Y": "TU1 A:00_0_R COMB Comdty",
    "US5Y": "FV1 A:00_0_R COMB Comdty",
    "US10Y": "TY1 A:00_0_R COMB Comdty",
    "US30Y": "WN1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # US nominal Treasury yields
    # Unit = %
    # Example:
    # 4.50 -> 4.50%
    # 0.01 change -> 1bp
    # --------------------------------------------------------
    "US2Y_YIELD": "USGG2YR Index",
    "US5Y_YIELD": "USGG5YR Index",
    "US10Y_YIELD": "USGG10YR Index",
    "US30Y_YIELD": "USGG30YR Index",

    # --------------------------------------------------------
    # US real yields (TIPS)
    # Unit = %
    # --------------------------------------------------------
    "US5Y_REAL": "USGGT05Y Index",
    "US10Y_REAL": "USGGT10Y Index",

    # --------------------------------------------------------
    # US breakeven inflation
    # Unit = %
    # --------------------------------------------------------
    "US5Y_BE": "USGGBE05 Index",
    "US10Y_BE": "USGGBE10 Index",

    # --------------------------------------------------------
    # European rates futures
    # --------------------------------------------------------
    "SCHATZ": "DU1 A:00_0_R COMB Comdty",
    "BOBL": "OE1 A:00_0_R COMB Comdty",
    "BUND": "RX1 A:00_0_R COMB Comdty",
    "BUXL": "UB1 A:00_0_R COMB Comdty",
    "BTP": "IK1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # Japanese rates futures
    # --------------------------------------------------------
    "JGB": "JB1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # FX spot
    # --------------------------------------------------------
    "USDJPY": "USDJPY BGN Curncy",
    "EURUSD": "EURUSD BGN Curncy",
    "GBPUSD": "GBPUSD BGN Curncy",
    "USDCNH": "USDCNH BGN Curncy",

    # --------------------------------------------------------
    # Energy
    # --------------------------------------------------------
    "WTI": "CL1 A:00_0_R COMB Comdty",
    "BRENT": "CO1 Comdty",

    # --------------------------------------------------------
    # Metals
    # --------------------------------------------------------
    "GOLD": "GC1 A:00_0_R COMB Comdty",
    "SILVER": "SI1 A:00_0_R COMB Comdty",
    "COPPER": "HG1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # Agriculture
    # --------------------------------------------------------
    "CORN": "C 1 A:00_0_R COMB Comdty",
    "WHEAT": "W 1 A:00_0_R COMB Comdty",
    "SOYBEAN": "S 1 A:00_0_R COMB Comdty",
}


# ============================================================
# Series Classification
#
# Useful for ChatGPT / Claude to distinguish:
# - tradable prices
# - nominal yields
# - real yields
# - breakeven inflation
# ============================================================

YIELD_IDS = {
    "US2Y_YIELD",
    "US5Y_YIELD",
    "US10Y_YIELD",
    "US30Y_YIELD",
}

REAL_YIELD_IDS = {
    "US5Y_REAL",
    "US10Y_REAL",
}

BREAKEVEN_IDS = {
    "US5Y_BE",
    "US10Y_BE",
}


def get_series_type(canonical_id):

    if canonical_id in YIELD_IDS:
        return "NOMINAL_YIELD"

    if canonical_id in REAL_YIELD_IDS:
        return "REAL_YIELD"

    if canonical_id in BREAKEVEN_IDS:
        return "BREAKEVEN"

    return "PRICE"


def get_unit(canonical_id):

    if (
        canonical_id in YIELD_IDS
        or canonical_id in REAL_YIELD_IDS
        or canonical_id in BREAKEVEN_IDS
    ):
        return "PCT"

    return "PRICE"


# ============================================================
# Function: Get Intraday Bars
# ============================================================

def get_intraday_bars(
    tickers,
    interval=1,
    tz="Asia/Tokyo",
):
    """
    Retrieve Bloomberg 1-minute intraday bars for:
        - yesterday
        - today

    Output includes:
        canonical_id
        series_type
        unit
        source_ticker
        request_date_jst
    """

    # --------------------------------------------------------
    # Dates
    # --------------------------------------------------------

    today = datetime.now(
        ZoneInfo(tz)
    ).date()

    yesterday = (
        today
        - timedelta(days=1)
    )

    dates = [
        yesterday.strftime("%Y-%m-%d"),
        today.strftime("%Y-%m-%d"),
    ]

    print("=" * 80)
    print("ICAM Bloomberg Intraday Download")
    print("=" * 80)

    print(f"Timezone    : {tz}")
    print(f"Dates       : {dates[0]} / {dates[1]}")
    print(f"Interval    : {interval} minute")
    print(f"Instruments : {len(tickers)}")

    print()

    # --------------------------------------------------------
    # Containers
    # --------------------------------------------------------

    dfs = []
    errors = []
    no_data = []

    # --------------------------------------------------------
    # Bloomberg requests
    # --------------------------------------------------------

    for canonical_id, ticker in tickers.items():

        series_type = get_series_type(
            canonical_id
        )

        unit = get_unit(
            canonical_id
        )

        for dt in dates:

            try:

                df = blp.bdib(
                    ticker=ticker,
                    dt=dt,
                    typ="TRADE",
                    interval=interval,
                    backend=Backend.PANDAS,
                    request_tz=tz,
                    output_tz=tz,
                )

                # ------------------------------------------------
                # No Data
                # ------------------------------------------------

                if df is None or df.empty:

                    no_data.append(
                        {
                            "canonical_id": canonical_id,
                            "ticker": ticker,
                            "series_type": series_type,
                            "date": dt,
                        }
                    )

                    print(
                        f"NO DATA | "
                        f"{canonical_id:<14} | "
                        f"{dt} | "
                        f"{ticker}"
                    )

                    continue

                # ------------------------------------------------
                # Add metadata
                # ------------------------------------------------

                df = df.copy()

                df["canonical_id"] = (
                    canonical_id
                )

                df["series_type"] = (
                    series_type
                )

                df["unit"] = (
                    unit
                )

                df["source_ticker"] = (
                    ticker
                )

                df["request_date_jst"] = (
                    dt
                )

                dfs.append(
                    df
                )

                print(
                    f"OK      | "
                    f"{canonical_id:<14} | "
                    f"{dt} | "
                    f"{len(df):>6,} rows"
                )

            except Exception as e:

                errors.append(
                    {
                        "canonical_id": canonical_id,
                        "ticker": ticker,
                        "series_type": series_type,
                        "date": dt,
                        "error": str(e),
                    }
                )

                print(
                    f"ERROR   | "
                    f"{canonical_id:<14} | "
                    f"{dt} | "
                    f"{e}"
                )

    # --------------------------------------------------------
    # Convert logs
    # --------------------------------------------------------

    error_df = pd.DataFrame(
        errors
    )

    no_data_df = pd.DataFrame(
        no_data
    )

    # --------------------------------------------------------
    # No successful requests
    # --------------------------------------------------------

    if not dfs:

        return (
            pd.DataFrame(),
            error_df,
            no_data_df,
        )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    bars = pd.concat(
        dfs,
        ignore_index=True,
    )

    # --------------------------------------------------------
    # Datetime
    # --------------------------------------------------------

    bars["time"] = pd.to_datetime(
        bars["time"]
    )

    # --------------------------------------------------------
    # Sort
    # --------------------------------------------------------

    bars = (
        bars
        .sort_values(
            [
                "time",
                "canonical_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return (
        bars,
        error_df,
        no_data_df,
    )


# ============================================================
# Run
# ============================================================

bars_1m, errors, no_data = (
    get_intraday_bars(
        tickers=TICKERS,
        interval=INTERVAL,
        tz=TZ,
    )
)


# ============================================================
# Summary
# ============================================================

print()
print("=" * 80)
print("ICAM RAW 2-DAY DATA SUMMARY")
print("=" * 80)

print(
    f"Total rows  : "
    f"{len(bars_1m):,}"
)


if bars_1m.empty:

    print(
        "No Bloomberg data was returned."
    )

else:

    # --------------------------------------------------------
    # Instrument count
    # --------------------------------------------------------

    instrument_count = (
        bars_1m[
            "canonical_id"
        ]
        .nunique()
    )

    print(
        f"Instruments : "
        f"{instrument_count}"
    )

    # --------------------------------------------------------
    # Time range
    # --------------------------------------------------------

    print(
        f"First time  : "
        f"{bars_1m['time'].min()}"
    )

    print(
        f"Last time   : "
        f"{bars_1m['time'].max()}"
    )

    # --------------------------------------------------------
    # Summary by instrument
    # --------------------------------------------------------

    summary = (
        bars_1m
        .groupby(
            [
                "canonical_id",
                "series_type",
                "unit",
            ]
        )
        .agg(
            rows=(
                "time",
                "size",
            ),
            first_time=(
                "time",
                "min",
            ),
            last_time=(
                "time",
                "max",
            ),
            first_value=(
                "close",
                "first",
            ),
            last_value=(
                "close",
                "last",
            ),
        )
        .sort_index()
    )

    print()
    print(
        "Rows by instrument:"
    )

    display(
        summary
    )


# ============================================================
# Data Quality Check
# ============================================================

if not bars_1m.empty:

    duplicate_count = (
        bars_1m
        .duplicated(
            subset=[
                "canonical_id",
                "time",
            ]
        )
        .sum()
    )

    missing_close_count = (
        bars_1m[
            "close"
        ]
        .isna()
        .sum()
    )

    invalid_price_count = (
        bars_1m.loc[
            bars_1m["series_type"] == "PRICE",
            "close",
        ]
        .le(0)
        .sum()
    )

    print()
    print("=" * 80)
    print("DATA QUALITY")
    print("=" * 80)

    print(
        f"Duplicate canonical_id/time : "
        f"{duplicate_count:,}"
    )

    print(
        f"Missing close               : "
        f"{missing_close_count:,}"
    )

    print(
        f"Invalid price <= 0           : "
        f"{invalid_price_count:,}"
    )


# ============================================================
# Optional Check:
# Nominal ≈ Real + Breakeven
# ============================================================

if not bars_1m.empty:

    rate_ids = [
        "US5Y_YIELD",
        "US5Y_REAL",
        "US5Y_BE",
        "US10Y_YIELD",
        "US10Y_REAL",
        "US10Y_BE",
    ]

    rate_check = (
        bars_1m[
            bars_1m[
                "canonical_id"
            ].isin(rate_ids)
        ]
        .pivot_table(
            index="time",
            columns="canonical_id",
            values="close",
            aggfunc="last",
        )
        .sort_index()
    )

    # --------------------------------------------------------
    # 5Y
    # --------------------------------------------------------

    required_5y = {
        "US5Y_YIELD",
        "US5Y_REAL",
        "US5Y_BE",
    }

    if required_5y.issubset(
        rate_check.columns
    ):

        rate_check[
            "US5Y_DECOMP_DIFF"
        ] = (
            rate_check[
                "US5Y_YIELD"
            ]
            - rate_check[
                "US5Y_REAL"
            ]
            - rate_check[
                "US5Y_BE"
            ]
        )

    # --------------------------------------------------------
    # 10Y
    # --------------------------------------------------------

    required_10y = {
        "US10Y_YIELD",
        "US10Y_REAL",
        "US10Y_BE",
    }

    if required_10y.issubset(
        rate_check.columns
    ):

        rate_check[
            "US10Y_DECOMP_DIFF"
        ] = (
            rate_check[
                "US10Y_YIELD"
            ]
            - rate_check[
                "US10Y_REAL"
            ]
            - rate_check[
                "US10Y_BE"
            ]
        )

    print()
    print("=" * 80)
    print("US RATES / INFLATION CHECK")
    print("=" * 80)

    display(
        rate_check.tail(10)
    )


# ============================================================
# Output CSV
# ============================================================

if not bars_1m.empty:

    # --------------------------------------------------------
    # File date = current JST date
    # --------------------------------------------------------

    today_str = (
        datetime.now(
            ZoneInfo(TZ)
        )
        .strftime(
            "%Y%m%d"
        )
    )

    # --------------------------------------------------------
    # Output filename
    # --------------------------------------------------------

    output_file = (
        OUTPUT_DIR
        / f"ICAM_Raw_2D_1m_{today_str}.csv"
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    bars_1m.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig",
    )

    # --------------------------------------------------------
    # File size
    # --------------------------------------------------------

    file_size_mb = (
        output_file.stat().st_size
        / 1024
        / 1024
    )

    print()
    print("=" * 80)
    print("CSV OUTPUT")
    print("=" * 80)

    print(
        f"File : "
        f"{output_file}"
    )

    print(
        f"Size : "
        f"{file_size_mb:.2f} MB"
    )


# ============================================================
# Errors
# ============================================================

if not errors.empty:

    print()
    print("=" * 80)
    print("ERRORS")
    print("=" * 80)

    display(
        errors
    )


# ============================================================
# No Data
# ============================================================

if not no_data.empty:

    print()
    print("=" * 80)
    print("NO DATA")
    print("=" * 80)

    display(
        no_data
    )

ICAM Bloomberg Intraday Download
Timezone    : Asia/Tokyo
Dates       : 2026-08-11 / 2026-08-12
Interval    : 1 minute
Instruments : 43

OK      | ES             | 2026-08-11 |  1,380 rows
OK      | ES             | 2026-08-12 |     68 rows
OK      | NQ             | 2026-08-11 |  1,380 rows
OK      | NQ             | 2026-08-12 |     68 rows
OK      | RTY            | 2026-08-11 |  1,332 rows
OK      | RTY            | 2026-08-12 |     68 rows
OK      | TOPIX          | 2026-08-11 |  1,050 rows
OK      | TOPIX          | 2026-08-12 |     68 rows
NO DATA | JREIT          | 2026-08-11 | TRE1 A:00_0_R COMB Index
NO DATA | JREIT          | 2026-08-12 | TRE1 A:00_0_R COMB Index
OK      | EURO           | 2026-08-11 |  1,085 rows
OK      | EURO           | 2026-08-12 |     68 rows
OK      | DAX            | 2026-08-11 |    874 rows
OK      | DAX            | 2026-08-12 |     67 rows
OK      | TAIWAN         | 2026-08-11 |  1,130 rows
OK      | TAIWAN         | 2026-08-12 |     56 rows
OK   

,,,rows,first_time,last_time,first_value,last_value
canonical_id,series_type,unit,,,,,
BOBL,PRICE,PRICE,911,2026-08-11 00:00:00+09:00,2026-08-12 01:07:00+09:00,113.970000,114.050000
BRENT,PRICE,PRICE,1331,2026-08-11 00:00:00+09:00,2026-08-12 00:57:00+09:00,86.100000,88.800000
BTP,PRICE,PRICE,726,2026-08-11 00:00:00+09:00,2026-08-12 01:07:00+09:00,116.650000,116.720000
BUND,PRICE,PRICE,1067,2026-08-11 00:00:00+09:00,2026-08-12 01:07:00+09:00,124.580000,124.740000
BUXL,PRICE,PRICE,917,2026-08-11 00:00:00+09:00,2026-08-12 01:07:00+09:00,106.180000,106.540000
CHINA_A50,PRICE,PRICE,1203,2026-08-11 00:00:00+09:00,2026-08-12 00:57:00+09:00,15004.000000,14925.000000
COPPER,PRICE,PRICE,1302,2026-08-11 00:00:00+09:00,2026-08-12 00:57:00+09:00,663.500000,663.350000
CORN,PRICE,PRICE,925,2026-08-11 00:00:00+09:00,2026-08-12 01:07:00+09:00,462.000000,460.000000
DAX,PRICE,PRICE,941,2026-08-11 00:00:00+09:00,2026-08-12 01:06:00+09:00,26421.000000,26486.000000



DATA QUALITY
Duplicate canonical_id/time : 0
Missing close               : 0
Invalid price <= 0           : 0

US RATES / INFLATION CHECK


canonical_id,US10Y_BE,US10Y_REAL,US10Y_YIELD,US5Y_BE,US5Y_REAL,US5Y_YIELD,US5Y_DECOMP_DIFF,US10Y_DECOMP_DIFF
time,,,,,,,,
2026-08-12 00:58:00+09:00,2.2532,2.4319,4.6863,2.2353,2.1412,4.3906,0.0141,0.0012
2026-08-12 00:59:00+09:00,2.2536,2.4310,4.6863,2.2334,2.1430,4.3888,0.0124,0.0017
2026-08-12 01:00:00+09:00,2.2529,2.4301,4.6842,2.2334,2.1412,4.3888,0.0142,0.0012
2026-08-12 01:01:00+09:00,2.2526,2.4283,4.6822,2.2333,2.1394,4.3870,0.0143,0.0013
2026-08-12 01:02:00+09:00,2.2515,2.4274,4.6802,2.2333,2.1394,4.3853,0.0126,0.0013
2026-08-12 01:03:00+09:00,2.2515,2.4274,4.6802,2.2333,2.1394,4.3870,0.0143,0.0013
2026-08-12 01:04:00+09:00,2.2529,2.4301,4.6842,2.2352,2.1412,4.3888,0.0124,0.0012
2026-08-12 01:05:00+09:00,2.2529,2.4301,4.6842,2.2343,2.1412,4.3888,0.0133,0.0012
2026-08-12 01:06:00+09:00,2.2520,2.4319,4.6842,2.2353,2.1412,4.3906,0.0141,0.0003



CSV OUTPUT
File : C:\Data\ICAM\ICAM_Raw_2D_1m_20260812.csv
Size : 6.43 MB

NO DATA


,canonical_id,ticker,series_type,date
0,JREIT,TRE1 A:00_0_R COMB Index,PRICE,2026-08-11
1,JREIT,TRE1 A:00_0_R COMB Index,PRICE,2026-08-12
2,JGB,JB1 A:00_0_R COMB Comdty,PRICE,2026-08-12
